# Step 14.3 — Pre-Norm vs Post-Norm：Normalization 到底放在 Residual 的哪一边？

Step 14.1 已经有 residual：

$$
x+F(x).
$$

Step 14.2 又引入 normalization：

$$
N(x).
$$

现在自然出现一个 architecture 问题：

> **N 应该放在 sublayer F 的前面，还是 residual addition 的后面？**

这就是 **Pre-Norm（前归一化）** 与 **Post-Norm（后归一化）** 的区别。

本节仍然只用一个 generic sublayer `F` 来讲结构。`F` 可以暂时理解成已经学过的 Multi-Head Attention；相同逻辑也会用于后面的其他 Transformer sublayers。

本节完成即停，不进入 Step 14.4 的 Feed-Forward MLP。


## 1. 先把两种公式写清楚

### Post-Norm

先计算 sublayer update，再 residual add，最后 normalize：

$$
\boxed{
y_{post}
=
N\big(x+F(x)\big)
}
$$

计算图：

```text
x
├──────── identity ───────┐
│                         │
└──── F(x) ───────────────┤
                          ↓
                       x + F(x)
                          ↓
                         N
                          ↓
                       y_post
```

### Pre-Norm

先 normalize，再把 normalized input 送进 sublayer；residual identity path 保留原始 x：

$$
\boxed{
y_{pre}
=
x+F\big(N(x)\big)
}
$$

计算图：

```text
x ───────────── identity ──────────────┐
│                                      │
└─ N(x) -> F(N(x)) ────────────────────┤
                                       ↓
                                x + F(N(x))
                                       ↓
                                    y_pre
```

二者 shape 都保持 `(B,T,C)`；差别不是 shape，而是 normalization 在 computation graph 中的位置。


## 2. Historical Path：Original Transformer 与后来的 Deep Pre-Norm Variants

原始 Transformer 的经典 **Add & Norm** 结构对应 Post-Norm 思路：

$$
N(x+F(x)).
$$

随着 Transformer 变得更深，实践中出现了大量把 normalization 移到 sublayer 前面的 **Pre-Norm variants**：

$$
x+F(N(x)).
$$

一个重要动机是改善深层网络的 optimization / gradient flow。

但这里不能写成：

> “Pre-Norm 在任何模型、任何深度、任何训练设定下都绝对优于 Post-Norm。”

更准确的说法是：

> **很多现代深层 Transformer / LLM 采用 Pre-Norm 或相关变体，因为它通常提供更直接的 residual gradient path，并在深层训练中表现出更友好的优化性质。**

Post-Norm 仍然是合法且有历史影响的重要结构。


## 3. 前向直觉：到底是谁被 Normalize？

### Post-Norm

$$
y=N(x+F(x)).
$$

这里 residual sum：

$$
x+F(x)
$$

整体都会进入 normalization。

所以 normalization 直接控制的是：

> residual addition 之后的 combined representation。

### Pre-Norm

$$
y=x+F(N(x)).
$$

这里 normalization 只进入 learned branch：

$$
N(x)\rightarrow F.
$$

而 residual identity branch 仍然直接传递：

$$
x.
$$

因此可以粗略记忆：

```text
Post-Norm: Add -> Norm
Pre-Norm : Norm -> Sublayer -> Add
```


## 4. 一个小例子：相同 x、相同 F、相同 RMSNorm，位置不同结果就不同

设：

$$
x=[1,2].
$$

使用 RMSNorm，先暂时忽略极小 epsilon，gamma 取初始值 1。

所以：

$$
N(x)
=
\frac{[1,2]}{\sqrt{(1^2+2^2)/2}}
=
\frac{[1,2]}{\sqrt{2.5}}
\approx
[0.6325,1.2649].
$$

定义一个固定线性教学 sublayer：

$$
F(z)=zW,
\qquad
W=
\begin{bmatrix}
0&1\\
-1&0
\end{bmatrix}.
$$

于是：

$$
F([a,b])=[-b,a].
$$

### Pre-Norm

$$
F(N(x))
\approx
[-1.2649,0.6325].
$$

所以：

$$
y_{pre}
=
[1,2]+[-1.2649,0.6325]
\approx
\boxed{[-0.2649,2.6325]}.
$$

### Post-Norm

先：

$$
F(x)=[-2,1].
$$

Residual sum：

$$
x+F(x)=[-1,3].
$$

再 RMSNorm：

$$
y_{post}
=
\frac{[-1,3]}{\sqrt{((-1)^2+3^2)/2}}
=
\frac{[-1,3]}{\sqrt5}
\approx
\boxed{[-0.4472,1.3416]}.
$$

所以即使 N、F、x 完全相同，只改变 normalization placement，计算函数本身就已经不同。


In [ ]:
import mlx.core as mx
import mlx.nn as nn

x = mx.array([[1.0, 2.0]])  # one token, C=2

# Fixed teaching transform: F([a,b]) = [-b,a]
W = mx.array([
    [ 0.0, 1.0],
    [-1.0, 0.0],
])

def F(z):
    return z @ W

norm = nn.RMSNorm(dims=2, eps=1e-5)

y_pre = x + F(norm(x))
y_post = norm(x + F(x))

print('x:', x)
print('pre-norm output :', y_pre)
print('post-norm output:', y_post)

# With gamma initialized to 1 and tiny eps, expected approximately:
# y_pre  ~= [-0.2649, 2.6325]
# y_post ~= [-0.4472, 1.3416]
# This master notebook remains unexecuted.


## 5. Pre-Norm 的 Gradient Path：Identity Route 真正绕过 N 与 F

Pre-Norm：

$$
y=x+F(N(x)).
$$

设：

$$
z=N(x).
$$

那么 chain rule：

$$
\frac{\partial F(N(x))}{\partial x}
=
J_F(z)J_N(x).
$$

因此：

$$
\boxed{
J_{pre}
=
I+J_FJ_N
}.
$$

用 column-gradient convention：

$$
\boxed{
g_x
=
g_y
+
J_N^T J_F^T g_y
}.
$$

关键是第一项：

$$
g_y.
$$

它直接来自 identity residual path，并且**不经过 normalization Jacobian，也不经过 sublayer Jacobian**。

这就是 Pre-Norm 在深层网络中 gradient flow 较友好的核心结构直觉。


## 6. Post-Norm 的 Gradient Path：Residual Sum 后仍必须经过 N

Post-Norm：

$$
y=N(x+F(x)).
$$

定义：

$$
u=x+F(x).
$$

那么：

$$
\frac{\partial u}{\partial x}
=
I+J_F.
$$

再经过 normalization：

$$
\boxed{
J_{post}
=
J_N(u)(I+J_F)
}.
$$

反向：

$$
\boxed{
g_x
=
(I+J_F)^T J_N^T g_y
}.
$$

注意这里不存在一个从最终 y 到 x、完全绕过 normalization 的纯 `g_y` 项。

Residual identity contribution 仍然存在于 `(I+J_F)` 中，但整个 gradient 先要通过 normalization Jacobian。

这就是 Pre-Norm / Post-Norm 在 gradient-path topology 上最重要的区别。


## 7. 为什么 Pre-Norm 常用于更深网络？以及边界是什么？

Pre-Norm 的：

$$
J_{pre}=I+J_FJ_N
$$

显式保留 identity term I。

所以很多层堆叠时，存在一条更直接的 residual route，通常有利于 optimization stability。

但这仍然不能推出：

- Pre-Norm 的 gradient 永远不会 vanish / explode；
- Pre-Norm 一定比 Post-Norm 得到更好的最终模型；
- normalization Jacobian 从此就不重要；
- residual branch 永远只是小 correction。

实际训练行为仍取决于初始化、深度、optimizer、learning rate、architecture、normalization variant 等。

因此“Pre-Norm 更适合深层训练”应该理解为经验与结构上的优化优势，而不是无条件数学定理。


## 8. 连接到我们之后的 Block Code：先只看 Generic Sublayer

现在可以写出两个最小 skeleton。

### Pre-Norm skeleton

```python
def pre_norm_step(x, norm, sublayer):
    return x + sublayer(norm(x))
```

### Post-Norm skeleton

```python
def post_norm_step(x, norm, sublayer):
    return norm(x + sublayer(x))
```

这两段代码已经完整表达 placement difference。

但目前 `sublayer` 仍然是 generic placeholder。

我们还没有进入 Transformer Block 里的另一类核心 sublayer——Feed-Forward MLP；那是 Step 14.4。


In [ ]:
def pre_norm_step(x, norm, sublayer):
    normalized = norm(x)
    update = sublayer(normalized)
    return x + update


def post_norm_step(x, norm, sublayer):
    update = sublayer(x)
    residual_sum = x + update
    return norm(residual_sum)


B, T, C = 2, 4, 6
x_seq = mx.zeros((B, T, C))

# Shape-only toy sublayer; it introduces no new Step 14.4 MLP content.
def zero_update(z):
    return mx.zeros(z.shape)

norm_seq = nn.RMSNorm(dims=C)

pre_out = pre_norm_step(x_seq, norm_seq, zero_update)
post_out = post_norm_step(x_seq, norm_seq, zero_update)

print('input:', x_seq.shape)
print('pre :', pre_out.shape)
print('post:', post_out.shape)

# Both preserve (B,T,C).
# This master notebook remains unexecuted.


## 9. 一个简洁但准确的历史记忆

可以先用下面这条线索记忆：

```text
Original Transformer era
        ↓
Post-Norm / Add & Norm

Deeper Transformer / modern LLM practice
        ↓
many Pre-Norm or related variants
```

但不要把它写成“历史从错误版本升级到正确版本”。

Post-Norm 与 Pre-Norm 是不同 optimization geometry 的 architecture choices。

Pre-Norm 在深层训练中的普及，一个重要原因就是 residual identity path 更直接；但不同模型还会结合不同 normalization、residual scaling、初始化与训练策略。


## 10. Step 14.3 材料边界

- Post-Norm generic form：`y = N(x + F(x))`；Pre-Norm generic form：`y = x + F(N(x))`。
- 二者都保持 `(B,T,C)`；区别是 normalization 在 computation graph 中的位置，而不是 shape。
- Pre-Norm Jacobian：`J_pre = I + J_F J_N`，因此反向包含不经过 N/F 的 direct `g_y` identity contribution。
- Post-Norm Jacobian：`J_post = J_N (I + J_F)`；最终 gradient 到 x 仍需经过 normalization Jacobian。
- 这解释了为什么很多现代深层 Transformer / LLM 采用 Pre-Norm 或相关 variants，但不构成 Pre-Norm universally superior 或 gradient guaranteed stable 的证明。
- 原始 Transformer 的经典 Add & Norm 对应 Post-Norm historical pattern；后续深层模型中 Pre-Norm 广泛出现。
- 本节 skeleton 只使用 generic sublayer F；不实现完整 Transformer Block。
- 本轮只准备 Step 14.3；不进入 Step 14.4，不讲 `C -> 4C -> activation -> C` Feed-Forward MLP，也不比较 Attention 与 MLP 的职责。
